In [ ]:
import os
import pathlib
import zipfile
import pandas as pd
import numpy as np
import tensorflow as tf
tf.__version__

In [ ]:
# From challenge author repo
def package_predictions_for_submission(scores, threshold, output=pathlib.Path('./submission.zip')):
  
  filename_variable_pairs = {
    'threshold.txt': threshold,
    'scores.txt': scores,
  }

  # The most straightforward way to populate a zip file is with actual files on disk
  # Therefore, temporarily create each file in the working directory  
  for f, d in filename_variable_pairs.items():
    np.savetxt(f, d)

  # Create the zip file
  with zipfile.ZipFile(output, mode='w') as z:
    for f, _ in filename_variable_pairs.items():
      z.write(f)
  
  # ...the cleanup after ourselves to avoid clutter
  for f, _ in filename_variable_pairs.items():
    os.remove(f)

  print(f'Successfully created CodaBench submission file: `{output}`. Upload this zip file to CodaBench to complete your submission.')

In [ ]:
X_ref = np.load('XRef_AllStepsR_f32_norm.npy')
# X_ref = np.load('XRef_AllStepsR_f32_normMM.npy')
Y_ref = np.load('YRef_AllStepsR.npy')
X_probe = np.load('XProbe_AllStepsR_f32_norm.npy')
# X_probe = np.load('XProbe_AllStepsR_f32_normMM.npy')
Y_probe = np.load('YProbe_AllStepsR.npy')

In [ ]:
# import inception
# model = inception.get_model(n_classes=200, input_shape=(None, 75*40), reduce=[tf.keras.layers.GlobalAveragePooling1D()])
# model.load_weights('001_best_model_norm.weights.h5') # 002 - 005
# model = tf.keras.models.load_model('011_best_model_normMM.keras') # 006 - 007
# model = tf.keras.models.load_model('011_last_model_normMM.keras') # 008
# model = tf.keras.models.load_model('021_best_model.keras') # 009
# model = tf.keras.models.load_model('031_best_model.keras') # 010
# model = tf.keras.models.load_model('041_best_model.keras') # 011
# model = tf.keras.models.load_model('041_last_model.keras') # 012
# model = tf.keras.models.load_model('002_last_model.keras') # 016
# model = tf.keras.models.load_model('003_checkpoint_epoch_800.keras') # 018 (not sure)
# model = tf.keras.models.load_model('052_last_model.keras', safe_mode=False) # 017
import inceptionembed
model = inceptionembed.get_model(n_classes=200, input_shape=(None, 75*40), reduce=[tf.keras.layers.GlobalAveragePooling1D()])
# model.load_weights('053_last_model.keras') # 019 (not sure)
model.load_weights('054_best_model.keras') # 020

In [ ]:
embedding_model = tf.keras.Model(
    inputs=model.input,
    outputs=model.get_layer("global_average_pooling1d_1").output
)

In [ ]:
X_ = X_ref
X_ = X_.reshape(X_.shape[0], X_.shape[1], -1)
# X_ = np.expand_dims(X_, axis=-1)
ref_emb = embedding_model.predict(X_)

In [ ]:
ref_emb.shape

In [ ]:
X_ = X_probe
X_ = X_.reshape(X_.shape[0], X_.shape[1], -1)
# X_ = np.expand_dims(X_, axis=-1)
probe_embA = embedding_model.predict(X_[0:5000], batch_size=64)
probe_embB = embedding_model.predict(X_[5000:10000], batch_size=64)
probe_embC = embedding_model.predict(X_[10000:15000], batch_size=64)
probe_embD = embedding_model.predict(X_[15000:], batch_size=64)

In [ ]:
probe_emb = np.concatenate([probe_embA, probe_embB, probe_embC, probe_embD], axis=0)
probe_emb.shape

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Archives

## submission_018 & 019 & 020

In [ ]:
ref_emb = tf.math.l2_normalize(ref_emb, axis=1).numpy()
probe_emb = tf.math.l2_normalize(probe_emb, axis=1).numpy()

In [ ]:
ref_person = ref_emb.reshape(-1,2,128).mean(axis=1)
probe_person = probe_emb.reshape(-1,2,128).mean(axis=1)

In [ ]:
classes = np.unique(Y_ref)

prototypes = []
proto_labels = []
for c in classes:
    members = ref_person[Y_ref[:, 0] == c]
    
    # On garde ce qui est dans le 2xstd de l'average de l'embedding
    centroid = members.mean(axis=0, keepdims=True)
    dists = cosine_similarity(members, centroid)[:, 0]
    mask = dists > (dists.mean() - 2 * dists.std())
    members = members[mask]
    
    # Why not medoid : "sample closest to the centroid"
    centroid = members.mean(axis=0, keepdims=True)
    medoid_idx = cosine_similarity(members, centroid)[:, 0].argmax()
    prototypes.append(members[medoid_idx])
    proto_labels.append(c)

prototypes = np.array(prototypes)
proto_labels = np.array(proto_labels)

In [ ]:
# score sur la class claim
scores = []
for i in range(len(probe_person)):
    claim = Y_probe[i]

    idx = np.where(proto_labels == claim)[0][0]

    score = cosine_similarity(
        probe_person[i:i+1],
        prototypes[idx:idx+1]
    )[0,0]

    scores.append(score)

scores = np.array(scores)

print(f"Raw scores — min: {scores.min():.4f}, max: {scores.max():.4f}, mean: {scores.mean():.4f}")

In [ ]:
# z-score normalization par classe
classes_probe = np.unique(Y_probe[:, 0])
class_score_stats = {}

for c in classes_probe:
    mask = Y_probe[:, 0] == c
    class_scores = scores[mask]
    class_score_stats[c] = {
        'mean': class_scores.mean(),
        'std': class_scores.std()
    }

normalized_scores = np.array([
    (scores[i] - class_score_stats[Y_probe[i, 0]]['mean']) / (class_score_stats[Y_probe[i, 0]]['std'] + 1e-8)
    for i in range(len(scores))
])

normalized_scores -= normalized_scores.min()

print(f"Normalized scores — min: {normalized_scores.min():.4f}, max: {normalized_scores.max():.4f}, mean: {normalized_scores.mean():.4f}")

In [ ]:
threshold = np.array([1.7])
pred_known = normalized_scores > threshold
pred_known.mean()

In [ ]:
# package_predictions_for_submission(normalized_scores, threshold, output=pathlib.Path('./submission_018.zip'))
# package_predictions_for_submission(normalized_scores, threshold, output=pathlib.Path('./submission_019.zip'))
package_predictions_for_submission(normalized_scores, threshold, output=pathlib.Path('./submission_020.zip'))

## submission_016 & 017

In [ ]:
ref_emb = tf.math.l2_normalize(ref_emb, axis=1).numpy()
probe_emb = tf.math.l2_normalize(probe_emb, axis=1).numpy()

In [ ]:
ref_person = ref_emb.reshape(-1,2,128).mean(axis=1)
probe_person = probe_emb.reshape(-1,2,128).mean(axis=1)

In [ ]:
classes = np.unique(Y_ref)

prototypes = []
proto_labels = []
for c in classes:
    members = ref_person[Y_ref[:, 0] == c]
    
    # On garde ce qui est dans le 2xstd de l'average de l'embedding
    centroid = members.mean(axis=0, keepdims=True)
    dists = cosine_similarity(members, centroid)[:, 0]
    mask = dists > (dists.mean() - 2 * dists.std())
    members = members[mask]
    
    # Why not medoid : "sample closest to the centroid"
    centroid = members.mean(axis=0, keepdims=True)
    medoid_idx = cosine_similarity(members, centroid)[:, 0].argmax()
    prototypes.append(members[medoid_idx])
    proto_labels.append(c)

prototypes = np.array(prototypes)
proto_labels = np.array(proto_labels)

In [ ]:
# score sur la class claim
scores = []
for i in range(len(probe_person)):
    claim = Y_probe[i]

    idx = np.where(proto_labels == claim)[0][0]

    score = cosine_similarity(
        probe_person[i:i+1],
        prototypes[idx:idx+1]
    )[0,0]

    scores.append(score)

scores = np.array(scores)

print(f"Raw scores — min: {scores.min():.4f}, max: {scores.max():.4f}, mean: {scores.mean():.4f}")

In [ ]:
# z-score normalization par classe
classes_probe = np.unique(Y_probe[:, 0])
class_score_stats = {}

for c in classes_probe:
    mask = Y_probe[:, 0] == c
    class_scores = scores[mask]
    class_score_stats[c] = {
        'mean': class_scores.mean(),
        'std': class_scores.std()
    }

normalized_scores = np.array([
    (scores[i] - class_score_stats[Y_probe[i, 0]]['mean']) / (class_score_stats[Y_probe[i, 0]]['std'] + 1e-8)
    for i in range(len(scores))
])

normalized_scores -= normalized_scores.min()

print(f"Normalized scores — min: {normalized_scores.min():.4f}, max: {normalized_scores.max():.4f}, mean: {normalized_scores.mean():.4f}")

In [ ]:
threshold = np.array([2])
pred_known = normalized_scores > threshold
pred_known.mean()

In [ ]:
# package_predictions_for_submission(normalized_scores, threshold, output=pathlib.Path('./submission_016.zip'))
# package_predictions_for_submission(normalized_scores, threshold, output=pathlib.Path('./submission_017.zip'))
package_predictions_for_submission(normalized_scores, threshold, output=pathlib.Path('./submission_017-2.zip'))

## submission_015

In [ ]:
ref_emb = tf.math.l2_normalize(ref_emb, axis=1).numpy()
probe_emb = tf.math.l2_normalize(probe_emb, axis=1).numpy()

In [ ]:
ref_person = ref_emb.reshape(-1,2,128).mean(axis=1)
probe_person = probe_emb.reshape(-1,2,128).mean(axis=1)

In [ ]:
# embedding moyen par classe
classes = np.unique(Y_ref)

prototypes = []
proto_labels = []

for c in classes:
    proto = ref_person[Y_ref[:,0] == c].mean(axis=0)
    prototypes.append(proto)
    proto_labels.append(c)

prototypes = np.array(prototypes)
proto_labels = np.array(proto_labels)

In [ ]:
# score sur la class claim
scores = []
for i in range(len(probe_person)):
    claim = Y_probe[i]

    idx = np.where(proto_labels == claim)[0][0]

    score = cosine_similarity(
        probe_person[i:i+1],
        prototypes[idx:idx+1]
    )[0,0]

    scores.append(score)

scores = np.array(scores)

print(f"Raw scores — min: {scores.min():.4f}, max: {scores.max():.4f}, mean: {scores.mean():.4f}")

In [ ]:
# For each class, compute how all OTHER prototypes score against it
# This gives an impostor distribution grounded in ref, not probe

class_zt_stats = {}
for idx, c in enumerate(proto_labels):
    # All prototypes except the claimed class = impostors
    impostor_mask = proto_labels != c
    impostor_scores = cosine_similarity(
        prototypes[impostor_mask],
        prototypes[idx:idx+1]
    )[:, 0]
    class_zt_stats[c] = {
        'mean': impostor_scores.mean(),
        'std': impostor_scores.std()
    }

# Normalize each probe score using its claimed class impostor stats
normalized_scores = np.array([
    (scores[i] - class_zt_stats[Y_probe[i, 0]]['mean']) / (class_zt_stats[Y_probe[i, 0]]['std'] + 1e-8)
    for i in range(len(scores))
])

normalized_scores -= normalized_scores.min()

print(f"ZT-norm scores — min: {normalized_scores.min():.4f}, max: {normalized_scores.max():.4f}, mean: {normalized_scores.mean():.4f}")

In [ ]:
threshold = np.array([3.75])
pred_known = normalized_scores > threshold
pred_known.mean()

In [ ]:
package_predictions_for_submission(normalized_scores, threshold, output=pathlib.Path('./submission_015.zip'))

## submission_014

In [ ]:
ref_emb = tf.math.l2_normalize(ref_emb, axis=1).numpy()
probe_emb = tf.math.l2_normalize(probe_emb, axis=1).numpy()

In [ ]:
ref_person = ref_emb.reshape(-1,2,128).mean(axis=1)
probe_person = probe_emb.reshape(-1,2,128).mean(axis=1)

In [ ]:
classes = np.unique(Y_ref)

prototypes = []
proto_labels = []
for c in classes:
    members = ref_person[Y_ref[:, 0] == c]
    
    # On garde ce qui est dans le 2xstd de l'average de l'embedding
    centroid = members.mean(axis=0, keepdims=True)
    dists = cosine_similarity(members, centroid)[:, 0]
    mask = dists > (dists.mean() - 2 * dists.std())
    members = members[mask]
    
    # Why not medoid : "sample closest to the centroid"
    centroid = members.mean(axis=0, keepdims=True)
    medoid_idx = cosine_similarity(members, centroid)[:, 0].argmax()
    prototypes.append(members[medoid_idx])
    proto_labels.append(c)

prototypes = np.array(prototypes)
proto_labels = np.array(proto_labels)

In [ ]:
# score sur la class claim
scores = []
for i in range(len(probe_person)):
    claim = Y_probe[i]

    idx = np.where(proto_labels == claim)[0][0]

    score = cosine_similarity(
        probe_person[i:i+1],
        prototypes[idx:idx+1]
    )[0,0]

    scores.append(score)

scores = np.array(scores)

print(f"Raw scores — min: {scores.min():.4f}, max: {scores.max():.4f}, mean: {scores.mean():.4f}")

In [ ]:
# z-score normalization par classe
classes_probe = np.unique(Y_probe[:, 0])
class_score_stats = {}

for c in classes_probe:
    mask = Y_probe[:, 0] == c
    class_scores = scores[mask]
    class_score_stats[c] = {
        'mean': class_scores.mean(),
        'std': class_scores.std()
    }

normalized_scores = np.array([
    (scores[i] - class_score_stats[Y_probe[i, 0]]['mean']) / (class_score_stats[Y_probe[i, 0]]['std'] + 1e-8)
    for i in range(len(scores))
])

normalized_scores -= normalized_scores.min()

print(f"Normalized scores — min: {normalized_scores.min():.4f}, max: {normalized_scores.max():.4f}, mean: {normalized_scores.mean():.4f}")

In [ ]:
threshold = np.array([2.7])
pred_known = normalized_scores > threshold
pred_known.mean()

In [ ]:
package_predictions_for_submission(normalized_scores, threshold, output=pathlib.Path('./submission_014.zip'))

## submission_013

In [ ]:
ref_emb = tf.math.l2_normalize(ref_emb, axis=1).numpy()
probe_emb = tf.math.l2_normalize(probe_emb, axis=1).numpy()

In [ ]:
ref_person = ref_emb.reshape(-1,2,128).mean(axis=1)
probe_person = probe_emb.reshape(-1,2,128).mean(axis=1)

In [ ]:
# embedding moyen par classe
classes = np.unique(Y_ref)

prototypes = []
proto_labels = []

for c in classes:
    proto = ref_person[Y_ref[:,0] == c].mean(axis=0)
    prototypes.append(proto)
    proto_labels.append(c)

prototypes = np.array(prototypes)
proto_labels = np.array(proto_labels)

In [ ]:
# score sur la class claim
scores = []
for i in range(len(probe_person)):
    claim = Y_probe[i]

    idx = np.where(proto_labels == claim)[0][0]

    score = cosine_similarity(
        probe_person[i:i+1],
        prototypes[idx:idx+1]
    )[0,0]

    scores.append(score)

scores = np.array(scores)

print(f"Raw scores — min: {scores.min():.4f}, max: {scores.max():.4f}, mean: {scores.mean():.4f}")

In [ ]:
# z-score normalization par classe
classes_probe = np.unique(Y_probe[:, 0])
class_score_stats = {}

for c in classes_probe:
    mask = Y_probe[:, 0] == c
    class_scores = scores[mask]
    class_score_stats[c] = {
        'mean': class_scores.mean(),
        'std': class_scores.std()
    }

normalized_scores = np.array([
    (scores[i] - class_score_stats[Y_probe[i, 0]]['mean']) / (class_score_stats[Y_probe[i, 0]]['std'] + 1e-8)
    for i in range(len(scores))
])

normalized_scores -= normalized_scores.min()

print(f"Normalized scores — min: {normalized_scores.min():.4f}, max: {normalized_scores.max():.4f}, mean: {normalized_scores.mean():.4f}")

In [ ]:
threshold = np.array([2.75])
pred_known = normalized_scores > threshold
pred_known.mean()

In [ ]:
package_predictions_for_submission(normalized_scores, threshold, output=pathlib.Path('./submission_013.zip'))

## submission_011 & 012

004 with model 041 best and last.

In [ ]:
ref_emb = tf.math.l2_normalize(ref_emb, axis=1).numpy()
probe_emb = tf.math.l2_normalize(probe_emb, axis=1).numpy()

In [ ]:
ref_person = ref_emb.reshape(-1,2,128).mean(axis=1)
probe_person = probe_emb.reshape(-1,2,128).mean(axis=1)

In [ ]:
# embedding moyen par classe
classes = np.unique(Y_ref)

prototypes = []
proto_labels = []

for c in classes:
    proto = ref_person[Y_ref[:,0] == c].mean(axis=0)
    prototypes.append(proto)
    proto_labels.append(c)

prototypes = np.array(prototypes)
proto_labels = np.array(proto_labels)

In [ ]:
# score sur la class claim
scores = []
for i in range(len(probe_person)):
    claim = Y_probe[i]

    idx = np.where(proto_labels == claim)[0][0]

    score = cosine_similarity(
        probe_person[i:i+1],
        prototypes[idx:idx+1]
    )[0,0]

    scores.append(score)

scores = np.array(scores)

In [ ]:
scores.min(), scores.max(), scores.mean()

In [ ]:
threshold = np.array([0.665])
pred_known = scores > threshold
pred_known.mean()

In [ ]:
# package_predictions_for_submission(scores, threshold, output=pathlib.Path('./submission_011.zip'))
package_predictions_for_submission(scores, threshold, output=pathlib.Path('./submission_012.zip'))

## submission_010

004 with model 031, so now using image not flatted

In [ ]:
ref_emb = tf.math.l2_normalize(ref_emb, axis=1).numpy()
probe_emb = tf.math.l2_normalize(probe_emb, axis=1).numpy()

In [ ]:
ref_person = ref_emb.reshape(-1,2,128).mean(axis=1)
probe_person = probe_emb.reshape(-1,2,128).mean(axis=1)

In [ ]:
# embedding moyen par classe
classes = np.unique(Y_ref)

prototypes = []
proto_labels = []

for c in classes:
    proto = ref_person[Y_ref[:,0] == c].mean(axis=0)
    prototypes.append(proto)
    proto_labels.append(c)

prototypes = np.array(prototypes)
proto_labels = np.array(proto_labels)

In [ ]:
# score sur la class claim
scores = []
for i in range(len(probe_person)):
    claim = Y_probe[i]

    idx = np.where(proto_labels == claim)[0][0]

    score = cosine_similarity(
        probe_person[i:i+1],
        prototypes[idx:idx+1]
    )[0,0]

    scores.append(score)

scores = np.array(scores)

In [ ]:
scores.min(), scores.max(), scores.mean()

In [ ]:
threshold = np.array([0.728])
pred_known = scores > threshold
pred_known.mean()

In [ ]:
package_predictions_for_submission(scores, threshold, output=pathlib.Path('./submission_010.zip'))

## submission_009

004 with model 021

In [ ]:
ref_emb = tf.math.l2_normalize(ref_emb, axis=1).numpy()
probe_emb = tf.math.l2_normalize(probe_emb, axis=1).numpy()

In [ ]:
ref_person = ref_emb.reshape(-1,2,128).mean(axis=1)
probe_person = probe_emb.reshape(-1,2,128).mean(axis=1)

In [ ]:
# embedding moyen par classe
classes = np.unique(Y_ref)

prototypes = []
proto_labels = []

for c in classes:
    proto = ref_person[Y_ref[:,0] == c].mean(axis=0)
    prototypes.append(proto)
    proto_labels.append(c)

prototypes = np.array(prototypes)
proto_labels = np.array(proto_labels)

In [ ]:
# score sur la class claim
scores = []
for i in range(len(probe_person)):
    claim = Y_probe[i]

    idx = np.where(proto_labels == claim)[0][0]

    score = cosine_similarity(
        probe_person[i:i+1],
        prototypes[idx:idx+1]
    )[0,0]

    scores.append(score)

scores = np.array(scores)

In [ ]:
scores.min(), scores.max(), scores.mean()

In [ ]:
threshold = np.array([0.71])
pred_known = scores > threshold
pred_known.mean()

In [ ]:
package_predictions_for_submission(scores, threshold, output=pathlib.Path('./submission_009.zip'))

## submission_008

004 with MM last model

In [ ]:
ref_emb = tf.math.l2_normalize(ref_emb, axis=1).numpy()
probe_emb = tf.math.l2_normalize(probe_emb, axis=1).numpy()

In [ ]:
ref_person = ref_emb.reshape(-1,2,128).mean(axis=1)
probe_person = probe_emb.reshape(-1,2,128).mean(axis=1)

In [ ]:
# embedding moyen par classe
classes = np.unique(Y_ref)

prototypes = []
proto_labels = []

for c in classes:
    proto = ref_person[Y_ref[:,0] == c].mean(axis=0)
    prototypes.append(proto)
    proto_labels.append(c)

prototypes = np.array(prototypes)
proto_labels = np.array(proto_labels)

In [ ]:
# score sur la class claim
scores = []
for i in range(len(probe_person)):
    claim = Y_probe[i]

    idx = np.where(proto_labels == claim)[0][0]

    score = cosine_similarity(
        probe_person[i:i+1],
        prototypes[idx:idx+1]
    )[0,0]

    scores.append(score)

scores = np.array(scores)

In [ ]:
scores.min(), scores.max(), scores.mean()

In [ ]:
threshold = np.array([0.715])
pred_known = scores > threshold
pred_known.mean()

In [ ]:
package_predictions_for_submission(scores, threshold, output=pathlib.Path('./submission_008.zip'))

## submission_006 & 007

submission 004 and 005 but with MM best model and data

In [ ]:
ref_emb = tf.math.l2_normalize(ref_emb, axis=1).numpy()
probe_emb = tf.math.l2_normalize(probe_emb, axis=1).numpy()

In [ ]:
ref_person = ref_emb.reshape(-1,2,128).mean(axis=1)
probe_person = probe_emb.reshape(-1,2,128).mean(axis=1)

In [ ]:
scores = np.zeros(len(probe_person))

for i in range(len(probe_person)):
    claim = Y_probe[i, 0]

    refs = ref_person[Y_ref[:,0] == claim]

    sim = cosine_similarity(
        probe_person[i:i+1],
        refs
    )

    scores[i] = sim.max()

In [ ]:
scores.min(), scores.max(), scores.mean()

In [ ]:
threshold = np.array([0.72])
pred_known = scores > threshold
pred_known.mean()

In [ ]:
package_predictions_for_submission(scores, threshold, output=pathlib.Path('./submission_007.zip'))

006

In [ ]:
ref_emb = tf.math.l2_normalize(ref_emb, axis=1).numpy()
probe_emb = tf.math.l2_normalize(probe_emb, axis=1).numpy()

In [ ]:
ref_person = ref_emb.reshape(-1,2,128).mean(axis=1)
probe_person = probe_emb.reshape(-1,2,128).mean(axis=1)

In [ ]:
# embedding moyen par classe
classes = np.unique(Y_ref)

prototypes = []
proto_labels = []

for c in classes:
    proto = ref_person[Y_ref[:,0] == c].mean(axis=0)
    prototypes.append(proto)
    proto_labels.append(c)

prototypes = np.array(prototypes)
proto_labels = np.array(proto_labels)

In [ ]:
# score sur la class claim
scores = []
for i in range(len(probe_person)):
    claim = Y_probe[i]

    idx = np.where(proto_labels == claim)[0][0]

    score = cosine_similarity(
        probe_person[i:i+1],
        prototypes[idx:idx+1]
    )[0,0]

    scores.append(score)

scores = np.array(scores)

In [ ]:
scores.min(), scores.max(), scores.mean()

In [ ]:
threshold = np.array([0.715])
pred_known = scores > threshold
pred_known.mean()

In [ ]:
package_predictions_for_submission(scores, threshold, output=pathlib.Path('./submission_006.zip'))

## submission_005

Same as 004 but do not create an average embeding per ref user, search for best match.
Same warning, junk fast code, be careful if reusing.

In [ ]:
ref_emb = tf.math.l2_normalize(ref_emb, axis=1).numpy()
probe_emb = tf.math.l2_normalize(probe_emb, axis=1).numpy()

In [ ]:
ref_person = ref_emb.reshape(-1,2,128).mean(axis=1)
probe_person = probe_emb.reshape(-1,2,128).mean(axis=1)

In [ ]:
scores = np.zeros(len(probe_person))

for i in range(len(probe_person)):
    claim = Y_probe[i, 0]

    refs = ref_person[Y_ref[:,0] == claim]

    sim = cosine_similarity(
        probe_person[i:i+1],
        refs
    )

    scores[i] = sim.max()

In [ ]:
scores.min(), scores.max(), scores.mean()

In [ ]:
threshold = np.array([0.72])
pred_known = scores > threshold
pred_known.mean()

In [ ]:
package_predictions_for_submission(scores, threshold, output=pathlib.Path('./submission_005.zip'))

## submission_004

We add idea of check if user correspond to claim, not to all ref.

Warning : embedding average + score claim is quick / junk code, do not trust !

In [ ]:
ref_emb = tf.math.l2_normalize(ref_emb, axis=1).numpy()
probe_emb = tf.math.l2_normalize(probe_emb, axis=1).numpy()

In [ ]:
ref_person = ref_emb.reshape(-1,2,128).mean(axis=1)
probe_person = probe_emb.reshape(-1,2,128).mean(axis=1)

In [ ]:
# embedding moyen par classe
classes = np.unique(Y_ref)

prototypes = []
proto_labels = []

for c in classes:
    proto = ref_person[Y_ref[:,0] == c].mean(axis=0)
    prototypes.append(proto)
    proto_labels.append(c)

prototypes = np.array(prototypes)
proto_labels = np.array(proto_labels)

In [ ]:
# score sur la class claim
scores = []
for i in range(len(probe_person)):
    claim = Y_probe[i]

    idx = np.where(proto_labels == claim)[0][0]

    score = cosine_similarity(
        probe_person[i:i+1],
        prototypes[idx:idx+1]
    )[0,0]

    scores.append(score)

scores = np.array(scores)

In [ ]:
scores.min(), scores.max(), scores.mean()

In [ ]:
threshold = np.array([0.715])
pred_known = scores > threshold
pred_known.mean()

In [ ]:
package_predictions_for_submission(scores, threshold, output=pathlib.Path('./submission_004.zip'))

## submission_003
alternative 002 where merge of L/R is made before cosine similarity

In [ ]:
ref_emb = tf.math.l2_normalize(ref_emb, axis=1).numpy()
probe_emb = tf.math.l2_normalize(probe_emb, axis=1).numpy()

In [ ]:
ref_person = ref_emb.reshape(-1,2,128).mean(axis=1)
probe_person = probe_emb.reshape(-1,2,128).mean(axis=1)
sim = cosine_similarity(probe_person, ref_person)
best = sim.max(axis=1)

In [ ]:
best.min(), best.max(), best.mean()

In [ ]:
threshold = np.array([0.815])
pred_known = best > threshold
pred_known.mean()

In [ ]:
package_predictions_for_submission(best, threshold, output=pathlib.Path('./submission_003.zip'))

## submission_002
fix by merging left and right footsteps
(you can see in the last cell that I failed first time...)

In [ ]:
ref_emb = tf.math.l2_normalize(ref_emb, axis=1).numpy()
probe_emb = tf.math.l2_normalize(probe_emb, axis=1).numpy()

In [ ]:
sim = cosine_similarity(probe_emb, ref_emb)
best = sim.max(axis=1)

In [ ]:
best_person = best.reshape(-1, 2).mean(axis=1)
best_person.shape, best_person.min(), best_person.max(), best_person.mean()

In [ ]:
threshold = np.array([0.7925])
pred_known = best_person > threshold
pred_known.mean()

In [ ]:
package_predictions_for_submission(best_person, threshold, output=pathlib.Path('./submission_002-1.zip'))
# package_predictions_for_submission(best, threshold, output=pathlib.Path('./submission_002.zip'))

## submission_001
todo / error : I probably need to get back to 10k output, I forgot about 1 sample = 2 footsteps

In [ ]:
ref_emb = tf.math.l2_normalize(ref_emb, axis=1).numpy()
probe_emb = tf.math.l2_normalize(probe_emb, axis=1).numpy()

In [ ]:
sim = cosine_similarity(probe_emb, ref_emb)
best = sim.max(axis=1)

In [ ]:
best.min(), best.max(), best.mean()

In [ ]:
threshold = np.array([0.7925])
pred_known = best > threshold
pred_known.mean()

In [ ]:
package_predictions_for_submission(best, threshold, output=pathlib.Path('./submission_001.zip'))